In [1]:
# ── Section 1. 환경 세팅 ──────────────────────────────────────
import numpy as np
import tensorflow as tf
import random
import gc
import time
import os
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

IS_KAGGLE = os.path.exists('/kaggle')
BASE = 'D:/Deep_Learning/DeepGuard/'
PREP = '/kaggle/input/notebooks/hongsh184/test122/preprocessed/' if IS_KAGGLE else BASE + 'preprocessed/'
MODELS_OUT = '/kaggle/working/models/' if IS_KAGGLE else BASE + 'models/'

os.makedirs(MODELS_OUT, exist_ok=True)

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("환경 세팅 완료")


2026-04-08 14:02:00.409955: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775656920.644761      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775656920.712762      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775656921.308060      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775656921.308143      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775656921.308147      17 computation_placer.cc:177] computation placer alr

환경 세팅 완료


In [2]:
# ── Section 2. W20 데이터 로드 ────────────────────────────────
X_train  = np.load(PREP + 'X_train_w20.npy')
X_val    = np.load(PREP + 'X_val_w20.npy')
X_te_tue = np.load(PREP + 'X_te_tue_w20.npy')
X_te_wed = np.load(PREP + 'X_te_wed_w20.npy')
X_te_fri = np.load(PREP + 'X_te_fri_w20.npy')
y_te_tue = np.load(PREP + 'y_te_tue_w20.npy', allow_pickle=True)
y_te_wed = np.load(PREP + 'y_te_wed_w20.npy', allow_pickle=True)
y_te_fri = np.load(PREP + 'y_te_fri_w20.npy', allow_pickle=True)

TIMESTEPS = X_train.shape[1]
FEATURES  = X_train.shape[2]

print(f"X_train shape : {X_train.shape}")
print(f"X_val   shape : {X_val.shape}")
print("W20 데이터 로드 완료")

X_train shape : (238257, 20, 78)
X_val   shape : (26473, 20, 78)
W20 데이터 로드 완료


In [3]:
# ── Section 2-1. 데이터 검증 ──────────────────────────────────
print("\n========== 데이터 검증 ==========")
assert not np.isnan(X_train).any(), "❌ X_train에 NaN 존재!"
assert not np.isinf(X_train).any(), "❌ X_train에 Inf 존재!"
assert not np.isnan(X_val).any(),   "❌ X_val에 NaN 존재!"
print("NaN/Inf 검사 통과 ✅")
print(f"X_train 실제 min: {X_train.min()} / max: {X_train.max()}")
assert X_train.min() >= -1e-6 and X_train.max() <= 1.0 + 1e-6, "❌ 스케일링 범위 이상!"
print("스케일링 범위 검사 통과 ✅")
print(f"y_te_tue 고유값: {np.unique(y_te_tue)}")
print(f"y_te_wed 고유값: {np.unique(y_te_wed)}")
print(f"y_te_fri 고유값: {np.unique(y_te_fri)}")
assert TIMESTEPS == 20, "❌ TIMESTEPS 불일치!"
assert FEATURES  == 78, "❌ FEATURES 불일치!"
print(f"TIMESTEPS: {TIMESTEPS} ✅ / FEATURES: {FEATURES} ✅")
print("========== 모든 검증 통과 ✅ 학습 시작 가능 ==========\n")


========== 데이터 검증 ==========
NaN/Inf 검사 통과 ✅
X_train 실제 min: 0.0 / max: 1.0000000000000002
스케일링 범위 검사 통과 ✅
y_te_tue 고유값: ['BENIGN' 'FTP-Patator']
y_te_wed 고유값: ['BENIGN' 'DoS slowloris']
y_te_fri 고유값: ['BENIGN' 'DDoS']
TIMESTEPS: 20 ✅ / FEATURES: 78 ✅
========== 모든 검증 통과 ✅ 학습 시작 가능 ==========



In [4]:
# ── Section 3. 공통 함수 ──────────────────────────────────────
def get_threshold(model, X_val, percentile=95):
    recon = model.predict(X_val, verbose=0)
    mse   = np.mean(np.power(X_val - recon, 2), axis=(1, 2))
    return np.percentile(mse, percentile)

def evaluate(model, threshold, name):
    results = {}
    for X, y, label in [
        (X_te_tue, y_te_tue, 'BruteForce'),
        (X_te_wed, y_te_wed, 'DoS'),
        (X_te_fri, y_te_fri, 'DDoS'),
    ]:
        recon = model.predict(X, verbose=0)
        mse   = np.mean(np.power(X - recon, 2), axis=(1, 2))
        pred  = (mse > threshold).astype(int)
        true  = (y != 'BENIGN').astype(int)
        tp = np.sum((pred == 1) & (true == 1))
        fp = np.sum((pred == 1) & (true == 0))
        fn = np.sum((pred == 0) & (true == 1))
        recall    = tp / (tp + fn + 1e-9)
        precision = tp / (tp + fp + 1e-9)
        f1        = 2 * precision * recall / (precision + recall + 1e-9)
        results[label] = {'recall': recall, 'precision': precision, 'f1': f1}
        print(f"  {label} → Recall: {recall:.3f} / Precision: {precision:.3f} / F1: {f1:.3f}")
    return results

def build_lstm(units, timesteps, features):
    inp      = Input(shape=(timesteps, features))
    encoded  = LSTM(units, activation='tanh', return_sequences=False)(inp)
    repeated = RepeatVector(timesteps)(encoded)
    decoded  = LSTM(units, activation='tanh', return_sequences=True)(repeated)
    out      = TimeDistributed(Dense(features))(decoded)
    return Model(inp, out)

def build_bilstm(units, timesteps, features):
    inp      = Input(shape=(timesteps, features))
    encoded  = Bidirectional(LSTM(units, activation='tanh', return_sequences=False))(inp)
    repeated = RepeatVector(timesteps)(encoded)
    decoded  = Bidirectional(LSTM(units, activation='tanh', return_sequences=True))(repeated)
    out      = TimeDistributed(Dense(features))(decoded)
    return Model(inp, out)

print("공통 함수 정의 완료")

공통 함수 정의 완료


In [5]:
# ── Section 4. LSTM BASE (tanh) ───────────────────────────────
print("\n========== EXP1: LSTM BASE (tanh) ==========")
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

lstm_model = build_lstm(64, TIMESTEPS, FEATURES)
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

start = time.time()
lstm_hist = lstm_model.fit(
    X_train, X_train, epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=2
)
lstm_time = (time.time() - start) / len(lstm_hist.history['loss'])
lstm_threshold = get_threshold(lstm_model, X_val, 95)
print(f"LSTM 임계값: {lstm_threshold:.6f} | 에폭당 평균: {lstm_time:.2f}s")
lstm_results = evaluate(lstm_model, lstm_threshold, 'LSTM')
lstm_avg_recall = np.mean([lstm_results[k]['recall'] for k in lstm_results])
lstm_model.save(MODELS_OUT + 'lstm_tanh_base.keras')
np.save(MODELS_OUT + 'lstm_tanh_base_threshold.npy', np.array(lstm_threshold))
del lstm_model; gc.collect()
print("LSTM BASE 완료 + 저장!")



========== EXP1: LSTM BASE (tanh) ==========


2026-04-08 14:03:25.768792: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20, 78)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        36,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 20, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 20, 78)         │         5,070 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,702 (291.80 KB)

 Trainable params: 74,702 (291.80 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
931/931 - 77s - 82ms/step - loss: 0.0133 - val_loss: 0.0120
Epoch 2/100
931/931 - 71s - 76ms/step - loss: 0.0115 - val_loss: 0.0108
Epoch 3/100
931/931 - 70s - 76ms/step - loss: 0.0106 - val_loss: 0.0104
Epoch 4/100
931/931 - 68s - 73ms/step - loss: 0.0100 - val_loss: 0.0096
Epoch 5/100
931/931 - 70s - 75ms/step - loss: 0.0094 - val_loss: 0.0091
Epoch 6/100
931/931 - 69s - 74ms/step - loss: 0.0088 - val_loss: 0.0086
Epoch 7/100
931/931 - 71s - 76ms/step - loss: 0.0083 - val_loss: 0.0082
Epoch 8/100
931/931 - 71s - 77ms/step - loss: 0.0079 - val_loss: 0.0077
Epoch 9/100
931/931 - 71s - 77ms/step - loss: 0.0074 - val_loss: 0.0072
Epoch 10/100
931/931 - 71s - 77ms/step - loss: 0.0069 - val_loss: 0.0067
Epoch 11/100
931/931 - 71s - 76ms/step - loss: 0.0064 - val_loss: 0.0062
Epoch 12/100
931/931 - 72s - 77ms/step - loss: 0.0060 - val_loss: 0.0058
Epoch 13/100
931/931 - 71s - 77ms/step - loss: 0.0055 - val_loss: 0.0053
Epoch 14/100
931/931 - 72s - 77ms/step - loss: 0.0051 - val_

In [6]:
# ── Section 5. Bi-LSTM BASE (tanh) ───────────────────────────
print("\n========== EXP2: Bi-LSTM BASE (tanh) ==========")
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

bilstm_model = build_bilstm(64, TIMESTEPS, FEATURES)
bilstm_model.compile(optimizer='adam', loss='mse')
bilstm_model.summary()

start = time.time()
bilstm_hist = bilstm_model.fit(
    X_train, X_train, epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=2
)
bilstm_time = (time.time() - start) / len(bilstm_hist.history['loss'])
bilstm_threshold = get_threshold(bilstm_model, X_val, 95)
print(f"Bi-LSTM 임계값: {bilstm_threshold:.6f} | 에폭당 평균: {bilstm_time:.2f}s")
bilstm_results = evaluate(bilstm_model, bilstm_threshold, 'Bi-LSTM')
bilstm_avg_recall = np.mean([bilstm_results[k]['recall'] for k in bilstm_results])
bilstm_model.save(MODELS_OUT + 'bilstm_tanh_base.keras')
np.save(MODELS_OUT + 'bilstm_tanh_base_threshold.npy', np.array(bilstm_threshold))
del bilstm_model; gc.collect()
print("Bi-LSTM BASE 완료 + 저장!")


========== EXP2: Bi-LSTM BASE (tanh) ==========


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20, 78)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        73,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 20, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 78)         │        10,062 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 182,094 (711.30 KB)

 Trainable params: 182,094 (711.30 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
931/931 - 128s - 137ms/step - loss: 0.0123 - val_loss: 0.0111
Epoch 2/100
931/931 - 119s - 128ms/step - loss: 0.0104 - val_loss: 0.0097
Epoch 3/100
931/931 - 117s - 126ms/step - loss: 0.0090 - val_loss: 0.0084
Epoch 4/100
931/931 - 117s - 125ms/step - loss: 0.0077 - val_loss: 0.0070
Epoch 5/100
931/931 - 123s - 132ms/step - loss: 0.0063 - val_loss: 0.0057
Epoch 6/100
931/931 - 121s - 130ms/step - loss: 0.0053 - val_loss: 0.0048
Epoch 7/100
931/931 - 121s - 130ms/step - loss: 0.0045 - val_loss: 0.0041
Epoch 8/100
931/931 - 119s - 128ms/step - loss: 0.0039 - val_loss: 0.0037
Epoch 9/100
931/931 - 120s - 129ms/step - loss: 0.0035 - val_loss: 0.0034
Epoch 10/100
931/931 - 118s - 127ms/step - loss: 0.0032 - val_loss: 0.0031
Epoch 11/100
931/931 - 120s - 128ms/step - loss: 0.0029 - val_loss: 0.0028
Epoch 12/100
931/931 - 117s - 126ms/step - loss: 0.0027 - val_loss: 0.0026
Epoch 13/100
931/931 - 118s - 126ms/step - loss: 0.0025 - val_loss: 0.0024
Epoch 14/100
931/931 - 118s - 126m

In [7]:
# ── Section 6. 승자 선정 ──────────────────────────────────────
print("\n========== LSTM vs Bi-LSTM (tanh) 비교 ==========")
print(f"{'모델':<10} {'평균Recall':>10} {'BruteForce':>12} {'DoS':>8} {'DDoS':>8}")
print("-" * 50)
print(f"{'LSTM':<10} {lstm_avg_recall:>10.3f} "
      f"{lstm_results['BruteForce']['recall']:>12.3f} "
      f"{lstm_results['DoS']['recall']:>8.3f} "
      f"{lstm_results['DDoS']['recall']:>8.3f}")
print(f"{'Bi-LSTM':<10} {bilstm_avg_recall:>10.3f} "
      f"{bilstm_results['BruteForce']['recall']:>12.3f} "
      f"{bilstm_results['DoS']['recall']:>8.3f} "
      f"{bilstm_results['DDoS']['recall']:>8.3f}")

if bilstm_avg_recall >= lstm_avg_recall:
    winner = 'Bi-LSTM'
    print(f"\n✅ 2차 방어선 승자: Bi-LSTM (평균 Recall {bilstm_avg_recall:.3f})")
    # Bi-LSTM 특성: units=64 → 실제 128노드 (양방향 2배)
    # → 압축 효과를 보려면 units=32 (실제 64)로 절반 감소
    tuning_configs = [
        (64, 97, 'bilstm', 'BILSTM_TUN_A_u64_p97'),  # 정밀도 단독 강화
        (32, 95, 'bilstm', 'BILSTM_TUN_B_u32_p95'),  # 압축 단독 효과
        (32, 97, 'bilstm', 'BILSTM_TUN_C_u32_p97'),  # 압축 + 정밀도
    ]
else:
    winner = 'LSTM'
    print(f"\n✅ 2차 방어선 승자: LSTM (평균 Recall {lstm_avg_recall:.3f})")
    # LSTM 특성: units=64 → 실제 64노드 (단방향)
    # → 압축 효과를 보려면 units=32로 절반 감소
    tuning_configs = [
        (64, 97, 'lstm', 'LSTM_TUN_A_u64_p97'),  # 정밀도 단독 강화
        (32, 95, 'lstm', 'LSTM_TUN_B_u32_p95'),  # 압축 단독 효과
        (32, 97, 'lstm', 'LSTM_TUN_C_u32_p97'),  # 압축 + 정밀도
    ]


========== LSTM vs Bi-LSTM (tanh) 비교 ==========
모델           평균Recall   BruteForce      DoS     DDoS
--------------------------------------------------
LSTM            0.564        0.170    0.637    0.885
Bi-LSTM         0.573        0.135    0.681    0.905

✅ 2차 방어선 승자: Bi-LSTM (평균 Recall 0.573)


In [8]:
# ── Section 7. 승자 모델 튜닝 ─────────────────────────────────
print(f"\n========== {winner} 심층 튜닝 시작 ==========")
tuning_results = {}

for units, percentile, model_type, name in tuning_configs:
    print(f"\n========== {name} 학습 시작 ==========")
    random.seed(42); np.random.seed(42); tf.random.set_seed(42)

    if model_type == 'bilstm':
        model = build_bilstm(units, TIMESTEPS, FEATURES)
    else:
        model = build_lstm(units, TIMESTEPS, FEATURES)

    model.compile(optimizer='adam', loss='mse')

    start = time.time()
    hist  = model.fit(
        X_train, X_train, epochs=100, batch_size=256,
        validation_data=(X_val, X_val),
        callbacks=[es], shuffle=True, verbose=2
    )
    elapsed = time.time() - start

    threshold = get_threshold(model, X_val, percentile)
    print(f"임계값: {threshold:.6f} | 에폭: {len(hist.history['loss'])} | 총 소요: {elapsed:.1f}s")

    results = evaluate(model, threshold, name)
    tuning_results[name] = {
        'units': units,
        'percentile': percentile,
        'threshold': threshold,
        'epochs': len(hist.history['loss']),
        'results': results
    }

    model.save(MODELS_OUT + f'{name}.keras')
    np.save(MODELS_OUT + f'{name}_threshold.npy', np.array(threshold))
    del model; gc.collect()
    print(f"{name} 저장 완료!")


========== Bi-LSTM 심층 튜닝 시작 ==========

========== BILSTM_TUN_A_u64_p97 학습 시작 ==========
Epoch 1/100
931/931 - 130s - 140ms/step - loss: 0.0123 - val_loss: 0.0111
Epoch 2/100
931/931 - 130s - 140ms/step - loss: 0.0104 - val_loss: 0.0097
Epoch 3/100
931/931 - 118s - 127ms/step - loss: 0.0090 - val_loss: 0.0084
Epoch 4/100
931/931 - 117s - 126ms/step - loss: 0.0077 - val_loss: 0.0070
Epoch 5/100
931/931 - 123s - 132ms/step - loss: 0.0063 - val_loss: 0.0057
Epoch 6/100
931/931 - 120s - 129ms/step - loss: 0.0053 - val_loss: 0.0048
Epoch 7/100
931/931 - 115s - 124ms/step - loss: 0.0045 - val_loss: 0.0041
Epoch 8/100
931/931 - 115s - 124ms/step - loss: 0.0039 - val_loss: 0.0037
Epoch 9/100
931/931 - 120s - 129ms/step - loss: 0.0035 - val_loss: 0.0034
Epoch 10/100
931/931 - 119s - 128ms/step - loss: 0.0032 - val_loss: 0.0031
Epoch 11/100
931/931 - 120s - 129ms/step - loss: 0.0029 - val_loss: 0.0028
Epoch 12/100
931/931 - 118s - 127ms/step - loss: 0.0027 - val_loss: 0.0026
Epoch 13/100
931/93

In [9]:
# ── Section 8. 최종 비교 출력 ─────────────────────────────────
print(f"\n========== {winner} 튜닝 최종 결과 ==========")
print(f"{'모델':<30} {'평균Recall':>10} {'BruteForce':>12} {'DoS':>8} {'DDoS':>8} {'평균Precision':>14}")
print("-" * 90)

# BASE 결과 출력
if winner == 'Bi-LSTM':
    base_avg_r = bilstm_avg_recall
    base_r = bilstm_results
    base_name = 'Bi-LSTM BASE (tanh)'
else:
    base_avg_r = lstm_avg_recall
    base_r = lstm_results
    base_name = 'LSTM BASE (tanh)'

base_avg_p = np.mean([base_r[k]['precision'] for k in base_r])
print(f"{base_name:<30} {base_avg_r:>10.3f} "
      f"{base_r['BruteForce']['recall']:>12.3f} "
      f"{base_r['DoS']['recall']:>8.3f} "
      f"{base_r['DDoS']['recall']:>8.3f} "
      f"{base_avg_p:>14.3f}")

for name, r in tuning_results.items():
    avg_recall    = np.mean([r['results'][k]['recall']    for k in r['results']])
    avg_precision = np.mean([r['results'][k]['precision'] for k in r['results']])
    print(f"{name:<30} {avg_recall:>10.3f} "
          f"{r['results']['BruteForce']['recall']:>12.3f} "
          f"{r['results']['DoS']['recall']:>8.3f} "
          f"{r['results']['DDoS']['recall']:>8.3f} "
          f"{avg_precision:>14.3f}")

print("\n모든 학습 및 평가 완료!")


========== Bi-LSTM 튜닝 최종 결과 ==========
모델                               평균Recall   BruteForce      DoS     DDoS    평균Precision
------------------------------------------------------------------------------------------
Bi-LSTM BASE (tanh)                 0.573        0.135    0.681    0.905          0.384
BILSTM_TUN_A_u64_p97                0.531        0.091    0.598    0.905          0.399
BILSTM_TUN_B_u32_p95                0.475        0.142    0.392    0.890          0.389
BILSTM_TUN_C_u32_p97                0.432        0.090    0.335    0.872          0.404

모든 학습 및 평가 완료!
